# Feature Engineering - Session 2.7

## Evidence-Based Feature Engineering for ML Modeling

**Session:** 2.7  
**Duration:** 6-8 hours  
**Objective:** Create derived features to enhance model predictive power

**What we'll create:**
1. **Clinical feature combinations** (ER+/HER2-, Triple Negative, etc.)
2. **Risk stratification features** (stage + grade combinations)
3. **Pathway × Clinical interactions** (ER status × Estrogen pathway)
4. **Feature importance analysis** (identify most predictive features)
5. **Correlation analysis** (remove redundant features)

**Input:** `merged_dataset_clean.csv` (from Session 2.6)  
**Output:** Enhanced feature set ready for final dataset preparation

**Evidence base:** Feature engineering best practices for clinical ML (Kourou et al. 2015, Bektaş et al. 2023)

Let's build powerful features! 🚀

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
project_dir = Path(r'D:\Projects\tcga-metabric-treatment-ai')
data_dir = project_dir / 'data' / 'merged'
results_dir = project_dir / 'results'
figures_dir = results_dir / 'figures' / 'feature_engineering'
figures_dir.mkdir(parents=True, exist_ok=True)

# Load clean dataset from Session 2.6
print("="*70)
print("SESSION 2.7: FEATURE ENGINEERING")
print("="*70)

print("\nLoading clean dataset from Session 2.6...")
df = pd.read_csv(data_dir / 'merged_dataset_clean.csv')

print(f"\nDataset loaded: {df.shape}")
print(f"  Patients: {df.shape[0]}")
print(f"  Features: {df.shape[1]}")

# Identify feature types
clinical_base = ['patient_id', 'cohort', 'age', 'race', 'er_status', 'pr_status', 
                 'her2_status', 'grade', 'stage', 'tumor_size', 'lymph_nodes_positive',
                 'os_days', 'os_status', 'rfs_days', 'rfs_status', 'pam50_subtype',
                 'chemotherapy', 'hormone_therapy', 'radiation_therapy']

imputed_vars = ['stage_harmonized', 'stage_imputed', 'lymph_nodes_imputed']
indicator_vars = ['race_missing', 'grade_missing', 'tumor_size_missing']
pathway_vars = [col for col in df.columns if col not in clinical_base + imputed_vars + indicator_vars]

print(f"\nFeature breakdown:")
print(f"  Clinical base: {len(clinical_base)}")
print(f"  Imputed variables: {len(imputed_vars)}")
print(f"  Missing indicators: {len(indicator_vars)}")
print(f"  Pathway scores: {len(pathway_vars)}")

print("\n✅ Data loaded successfully!")
print("   Ready for feature engineering")

SESSION 2.7: FEATURE ENGINEERING

Loading clean dataset from Session 2.6...

Dataset loaded: (3075, 101)
  Patients: 3075
  Features: 101

Feature breakdown:
  Clinical base: 19
  Imputed variables: 3
  Missing indicators: 3
  Pathway scores: 76

✅ Data loaded successfully!
   Ready for feature engineering


### Part 1: Clinical Feature Combinations

**Objective:** Create clinically meaningful derived features

**Features to create:**
1. **Molecular subtypes** (ER+/HER2-, Triple Negative, etc.)
2. **Risk groups** (stage × grade combinations)
3. **Biomarker combinations** (receptor status patterns)
4. **Treatment response groups** (based on subtype + treatment)

**Clinical rationale:**
- ER+/HER2- = Luminal (hormone therapy candidates)
- Triple Negative = Aggressive, chemo-sensitive
- High risk = Stage III-IV + Grade 3

In [4]:
# Part 1: Clinical Feature Combinations (FIXED)
print("="*70)
print("PART 1: CLINICAL FEATURE COMBINATIONS")
print("="*70)

# Create working copy
df_features = df.copy()

# First: Clean grade variable for use in risk stratification
print("🔧 Preprocessing: Convert grade to numeric")
df_features['grade_numeric'] = pd.to_numeric(df_features['grade'], errors='coerce')
print(f"   Grade values: {df_features['grade_numeric'].dropna().unique()}")

# Feature 1: Molecular Subtype Groups (clinically important)
print("\n" + "="*70)
print("1. MOLECULAR SUBTYPE GROUPS")

def get_molecular_subtype(row):
    """Clinical molecular subtype classification"""
    er = row['er_status']
    pr = row['pr_status']
    her2 = row['her2_status']
    
    # Triple Negative
    if er == 'Negative' and pr == 'Negative' and her2 == 'Negative':
        return 'Triple_Negative'
    # HER2+ (regardless of hormone status)
    elif her2 == 'Positive':
        return 'HER2_Positive'
    # ER+ and/or PR+ (HER2-)
    elif (er == 'Positive' or pr == 'Positive') and her2 == 'Negative':
        return 'Hormone_Positive'
    # Unknown (missing data)
    else:
        return 'Unknown'

df_features['molecular_subtype'] = df_features.apply(get_molecular_subtype, axis=1)

print("Distribution:")
print(df_features['molecular_subtype'].value_counts())

# Feature 2: Risk Groups (stage + grade)
print("\n" + "="*70)
print("2. RISK STRATIFICATION GROUPS")

def get_risk_group(row):
    """Risk stratification based on stage and grade"""
    stage = row['stage_imputed']
    grade = row['grade_numeric']
    
    # High risk: Stage III-IV or Grade 3
    if stage >= 3 or (pd.notna(grade) and grade >= 3):
        return 'High_Risk'
    # Low risk: Stage 0-I and Grade 1
    elif stage <= 1 and (pd.notna(grade) and grade <= 1):
        return 'Low_Risk'
    # Intermediate
    else:
        return 'Intermediate_Risk'

df_features['risk_group'] = df_features.apply(get_risk_group, axis=1)

print("Distribution:")
print(df_features['risk_group'].value_counts())

# Feature 3: Nodal Status Binary
print("\n" + "="*70)
print("3. LYMPH NODE STATUS (BINARY)")

df_features['node_positive'] = (df_features['lymph_nodes_imputed'] > 0).astype(int)

print("Distribution:")
print(df_features['node_positive'].value_counts())

# Feature 4: Age Groups
print("\n" + "="*70)
print("4. AGE GROUPS")

def get_age_group(age):
    """Age stratification"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 40:
        return 'Young'
    elif age < 55:
        return 'Middle'
    elif age < 70:
        return 'Older'
    else:
        return 'Elderly'

df_features['age_group'] = df_features['age'].apply(get_age_group)

print("Distribution:")
print(df_features['age_group'].value_counts())

# Feature 5: Treatment Combination
print("\n" + "="*70)
print("5. TREATMENT COMBINATIONS")

def get_treatment_combo(row):
    """Treatment combination pattern"""
    chemo = row['chemotherapy'] in ['True', 'YES', True]
    hormone = row['hormone_therapy'] in ['True', 'YES', True]
    radiation = row['radiation_therapy'] in ['True', 'YES', True]
    
    treatments = []
    if chemo: treatments.append('C')
    if hormone: treatments.append('H')
    if radiation: treatments.append('R')
    
    if not treatments:
        return 'No_Treatment'
    else:
        return '+'.join(sorted(treatments))

df_features['treatment_combo'] = df_features.apply(get_treatment_combo, axis=1)

print("Distribution (top 10):")
print(df_features['treatment_combo'].value_counts().head(10))

# Summary
print("\n" + "="*70)
print("CLINICAL FEATURES CREATED")
print("="*70)

new_clinical_features = ['molecular_subtype', 'risk_group', 'node_positive', 
                         'age_group', 'treatment_combo']

print(f"\n✅ Created {len(new_clinical_features)} new clinical features:")
for feat in new_clinical_features:
    print(f"   • {feat}")

print(f"\nTotal features now: {df_features.shape[1]}")

PART 1: CLINICAL FEATURE COMBINATIONS
🔧 Preprocessing: Convert grade to numeric
   Grade values: [3. 2. 1.]

1. MOLECULAR SUBTYPE GROUPS
Distribution:
molecular_subtype
Hormone_Positive    1857
Triple_Negative      435
HER2_Positive        411
Unknown              372
Name: count, dtype: int64

2. RISK STRATIFICATION GROUPS
Distribution:
risk_group
Intermediate_Risk    1732
High_Risk            1273
Low_Risk               70
Name: count, dtype: int64

3. LYMPH NODE STATUS (BINARY)
Distribution:
node_positive
1    2068
0    1007
Name: count, dtype: int64

4. AGE GROUPS
Distribution:
age_group
Older      1228
Middle      891
Elderly     761
Young       195
Name: count, dtype: int64

5. TREATMENT COMBINATIONS
Distribution (top 10):
treatment_combo
H+R             798
R               535
C+H+R           450
H               421
C+R             400
No_Treatment    391
C                50
C+H              30
Name: count, dtype: int64

CLINICAL FEATURES CREATED

✅ Created 5 new clinical featur

### Part 2: Pathway × Clinical Interaction Features

**Objective:** Capture biology-driven interactions between pathways and clinical features

**Why this matters:**
- ER+ patients with HIGH estrogen pathway = likely responders to hormone therapy
- Triple Negative with HIGH proliferation = aggressive, needs chemo
- HER2+ with HIGH ERBB2 pathway = HER2-targeted therapy candidates

**Features to create:**
1. ER status × Estrogen pathway interaction
2. Grade × Proliferation pathway interaction
3. Molecular subtype × key pathway scores
4. High/low pathway activity groups

In [5]:
# Part 2: Pathway × Clinical Interaction Features
print("="*70)
print("PART 2: PATHWAY × CLINICAL INTERACTION FEATURES")
print("="*70)

# Identify key pathways for interactions
key_pathways = {
    'estrogen': 'HALLMARK_ESTROGEN_RESPONSE_EARLY',
    'proliferation': 'HALLMARK_E2F_TARGETS',
    'her2': 'HALLMARK_APICAL_JUNCTION',  # Often correlated with HER2
    'immune': 'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'glycolysis': 'HALLMARK_GLYCOLYSIS',
    'hypoxia': 'HALLMARK_HYPOXIA'
}

print(f"\nKey pathways for interaction features:")
for name, pathway in key_pathways.items():
    print(f"  • {name:15} → {pathway}")

# Interaction 1: ER status × Estrogen pathway
print("\n" + "="*70)
print("1. ER STATUS × ESTROGEN PATHWAY INTERACTION")

estrogen_pathway = key_pathways['estrogen']

def get_er_estrogen_interaction(row):
    """ER+ with high estrogen = strong hormone responsive"""
    er = row['er_status']
    estrogen_score = row[estrogen_pathway]
    
    if er == 'Positive' and estrogen_score > 0.5:
        return 'ER+_HighEstrogen'
    elif er == 'Positive' and estrogen_score <= 0.5:
        return 'ER+_LowEstrogen'
    elif er == 'Negative' and estrogen_score > 0:
        return 'ER-_UnexpectedEstrogen'
    elif er == 'Negative':
        return 'ER-_NoEstrogen'
    else:
        return 'Unknown'

df_features['er_estrogen_interaction'] = df_features.apply(get_er_estrogen_interaction, axis=1)

print("Distribution:")
print(df_features['er_estrogen_interaction'].value_counts())

# Interaction 2: Grade × Proliferation pathway
print("\n" + "="*70)
print("2. GRADE × PROLIFERATION PATHWAY INTERACTION")

proliferation_pathway = key_pathways['proliferation']

def get_grade_proliferation_interaction(row):
    """High grade + high proliferation = very aggressive"""
    grade = row['grade_numeric']
    prolif_score = row[proliferation_pathway]
    
    if pd.notna(grade):
        if grade >= 3 and prolif_score > 0.5:
            return 'HighGrade_HighProlif'
        elif grade >= 3:
            return 'HighGrade_LowProlif'
        elif grade <= 1 and prolif_score < -0.5:
            return 'LowGrade_LowProlif'
        elif grade <= 1:
            return 'LowGrade_HighProlif'
        else:
            return 'IntermediateGrade'
    else:
        return 'Unknown'

df_features['grade_proliferation_interaction'] = df_features.apply(get_grade_proliferation_interaction, axis=1)

print("Distribution:")
print(df_features['grade_proliferation_interaction'].value_counts())

# Interaction 3: Molecular subtype × pathway groups
print("\n" + "="*70)
print("3. TRIPLE NEGATIVE × GLYCOLYSIS INTERACTION")

glycolysis_pathway = key_pathways['glycolysis']

def get_tn_glycolysis_interaction(row):
    """Triple negative often has high glycolysis"""
    subtype = row['molecular_subtype']
    glycolysis_score = row[glycolysis_pathway]
    
    if subtype == 'Triple_Negative' and glycolysis_score > 0.5:
        return 'TN_HighGlycolysis'
    elif subtype == 'Triple_Negative':
        return 'TN_LowGlycolysis'
    else:
        return 'NotTN'

df_features['tn_glycolysis_interaction'] = df_features.apply(get_tn_glycolysis_interaction, axis=1)

print("Distribution:")
print(df_features['tn_glycolysis_interaction'].value_counts())

# Feature 4: High pathway activity count
print("\n" + "="*70)
print("4. PATHWAY ACTIVITY SUMMARY FEATURES")

# Count number of highly activated pathways (score > 1.0)
pathway_cols = [col for col in df_features.columns if 'HALLMARK' in col]

df_features['high_pathway_count'] = (df_features[pathway_cols] > 1.0).sum(axis=1)
df_features['low_pathway_count'] = (df_features[pathway_cols] < -1.0).sum(axis=1)
df_features['pathway_activity_ratio'] = df_features['high_pathway_count'] / (df_features['low_pathway_count'] + 1)

print(f"High pathway activity count:")
print(f"  Mean: {df_features['high_pathway_count'].mean():.2f}")
print(f"  Range: {df_features['high_pathway_count'].min():.0f} - {df_features['high_pathway_count'].max():.0f}")

print(f"\nLow pathway activity count:")
print(f"  Mean: {df_features['low_pathway_count'].mean():.2f}")
print(f"  Range: {df_features['low_pathway_count'].min():.0f} - {df_features['low_pathway_count'].max():.0f}")

# Summary
print("\n" + "="*70)
print("PATHWAY INTERACTION FEATURES CREATED")
print("="*70)

new_interaction_features = ['er_estrogen_interaction', 'grade_proliferation_interaction', 
                            'tn_glycolysis_interaction', 'high_pathway_count', 
                            'low_pathway_count', 'pathway_activity_ratio']

print(f"\n✅ Created {len(new_interaction_features)} pathway interaction features:")
for feat in new_interaction_features:
    print(f"   • {feat}")

print(f"\nTotal features now: {df_features.shape[1]}")

PART 2: PATHWAY × CLINICAL INTERACTION FEATURES

Key pathways for interaction features:
  • estrogen        → HALLMARK_ESTROGEN_RESPONSE_EARLY
  • proliferation   → HALLMARK_E2F_TARGETS
  • her2            → HALLMARK_APICAL_JUNCTION
  • immune          → HALLMARK_INTERFERON_GAMMA_RESPONSE
  • glycolysis      → HALLMARK_GLYCOLYSIS
  • hypoxia         → HALLMARK_HYPOXIA

1. ER STATUS × ESTROGEN PATHWAY INTERACTION


KeyError: 'HALLMARK_ESTROGEN_RESPONSE_EARLY'

In [6]:
# Debug: Check actual pathway column names
print("="*70)
print("DEBUGGING: CHECK ACTUAL PATHWAY COLUMN NAMES")
print("="*70)

# Get pathway columns
pathway_cols = [col for col in df_features.columns if col not in clinical_base + imputed_vars + indicator_vars + ['grade_numeric', 'molecular_subtype', 'risk_group', 'node_positive', 'age_group', 'treatment_combo']]

print(f"\nTotal pathway columns: {len(pathway_cols)}")
print(f"\nFirst 10 pathway column names:")
for i, col in enumerate(pathway_cols[:10], 1):
    print(f"  {i:2d}. {col}")

# Check if they contain "HALLMARK" or different naming
print(f"\nSample pathway names:")
if 'HALLMARK' in pathway_cols[0]:
    print("  Format: HALLMARK_...")
elif 'hallmark' in pathway_cols[0].lower():
    print("  Format: hallmark_... (lowercase)")
else:
    print(f"  Format: {pathway_cols[0][:50]}...")

# Search for estrogen-related pathways
print(f"\nSearching for estrogen-related pathways:")
estrogen_pathways = [col for col in pathway_cols if 'estrogen' in col.lower() or 'ESTROGEN' in col]
for pathway in estrogen_pathways:
    print(f"  • {pathway}")

# Search for E2F (proliferation) pathways
print(f"\nSearching for E2F/proliferation pathways:")
e2f_pathways = [col for col in pathway_cols if 'e2f' in col.lower() or 'E2F' in col]
for pathway in e2f_pathways:
    print(f"  • {pathway}")

DEBUGGING: CHECK ACTUAL PATHWAY COLUMN NAMES

Total pathway columns: 76

First 10 pathway column names:
   1. Adipogenesis
   2. Allograft Rejection
   3. Androgen Response
   4. Angiogenesis
   5. Apical Junction
   6. Apical Surface
   7. Apoptosis
   8. Autoimmune thyroid disease
   9. Bile Acid Metabolism
  10. Bladder cancer

Sample pathway names:
  Format: Adipogenesis...

Searching for estrogen-related pathways:
  • Estrogen Response Early
  • Estrogen Response Late
  • Estrogen signaling pathway

Searching for E2F/proliferation pathways:
  • E2F Targets


In [8]:
# Part 2: Pathway × Clinical Interaction Features (FIXED)
print("="*70)
print("PART 2: PATHWAY × CLINICAL INTERACTION FEATURES")
print("="*70)

# Identify key pathways for interactions (CORRECTED NAMES)
key_pathways = {
    'estrogen': 'Estrogen Response Early',
    'proliferation': 'E2F Targets',
    'immune': 'Interferon Gamma Response',
    'glycolysis': 'Glycolysis',
    'hypoxia': 'Hypoxia'
}

print(f"\nKey pathways for interaction features:")
for name, pathway in key_pathways.items():
    print(f"  • {name:15} → {pathway}")

# Interaction 1: ER status × Estrogen pathway
print("\n" + "="*70)
print("1. ER STATUS × ESTROGEN PATHWAY INTERACTION")

estrogen_pathway = key_pathways['estrogen']

def get_er_estrogen_interaction(row):
    """ER+ with high estrogen = strong hormone responsive"""
    er = row['er_status']
    estrogen_score = row[estrogen_pathway]
    
    if er == 'Positive' and estrogen_score > 0.5:
        return 'ER+_HighEstrogen'
    elif er == 'Positive' and estrogen_score <= 0.5:
        return 'ER+_LowEstrogen'
    elif er == 'Negative' and estrogen_score > 0:
        return 'ER-_UnexpectedEstrogen'
    elif er == 'Negative':
        return 'ER-_NoEstrogen'
    else:
        return 'Unknown'

df_features['er_estrogen_interaction'] = df_features.apply(get_er_estrogen_interaction, axis=1)

print("Distribution:")
print(df_features['er_estrogen_interaction'].value_counts())

# Interaction 2: Grade × Proliferation pathway
print("\n" + "="*70)
print("2. GRADE × PROLIFERATION PATHWAY INTERACTION")

proliferation_pathway = key_pathways['proliferation']

def get_grade_proliferation_interaction(row):
    """High grade + high proliferation = very aggressive"""
    grade = row['grade_numeric']
    prolif_score = row[proliferation_pathway]
    
    if pd.notna(grade):
        if grade >= 3 and prolif_score > 0.5:
            return 'HighGrade_HighProlif'
        elif grade >= 3:
            return 'HighGrade_LowProlif'
        elif grade <= 1 and prolif_score < -0.5:
            return 'LowGrade_LowProlif'
        elif grade <= 1:
            return 'LowGrade_HighProlif'
        else:
            return 'IntermediateGrade'
    else:
        return 'Unknown'

df_features['grade_proliferation_interaction'] = df_features.apply(get_grade_proliferation_interaction, axis=1)

print("Distribution:")
print(df_features['grade_proliferation_interaction'].value_counts())

# Interaction 3: Molecular subtype × glycolysis
print("\n" + "="*70)
print("3. TRIPLE NEGATIVE × GLYCOLYSIS INTERACTION")

glycolysis_pathway = key_pathways['glycolysis']

def get_tn_glycolysis_interaction(row):
    """Triple negative often has high glycolysis"""
    subtype = row['molecular_subtype']
    glycolysis_score = row[glycolysis_pathway]
    
    if subtype == 'Triple_Negative' and glycolysis_score > 0.5:
        return 'TN_HighGlycolysis'
    elif subtype == 'Triple_Negative':
        return 'TN_LowGlycolysis'
    else:
        return 'NotTN'

df_features['tn_glycolysis_interaction'] = df_features.apply(get_tn_glycolysis_interaction, axis=1)

print("Distribution:")
print(df_features['tn_glycolysis_interaction'].value_counts())

# Feature 4: High pathway activity count
print("\n" + "="*70)
print("4. PATHWAY ACTIVITY SUMMARY FEATURES")

# Get actual pathway columns
pathway_cols = [col for col in df_features.columns if col not in clinical_base + imputed_vars + indicator_vars + ['grade_numeric', 'molecular_subtype', 'risk_group', 'node_positive', 'age_group', 'treatment_combo']]

df_features['high_pathway_count'] = (df_features[pathway_cols] > 1.0).sum(axis=1)
df_features['low_pathway_count'] = (df_features[pathway_cols] < -1.0).sum(axis=1)
df_features['pathway_activity_ratio'] = df_features['high_pathway_count'] / (df_features['low_pathway_count'] + 1)

print(f"High pathway activity count:")
print(f"  Mean: {df_features['high_pathway_count'].mean():.2f}")
print(f"  Range: {df_features['high_pathway_count'].min():.0f} - {df_features['high_pathway_count'].max():.0f}")

print(f"\nLow pathway activity count:")
print(f"  Mean: {df_features['low_pathway_count'].mean():.2f}")
print(f"  Range: {df_features['low_pathway_count'].min():.0f} - {df_features['low_pathway_count'].max():.0f}")

# Summary
print("\n" + "="*70)
print("PATHWAY INTERACTION FEATURES CREATED")
print("="*70)

new_interaction_features = ['er_estrogen_interaction', 'grade_proliferation_interaction', 
                            'tn_glycolysis_interaction', 'high_pathway_count', 
                            'low_pathway_count', 'pathway_activity_ratio']

print(f"\n✅ Created {len(new_interaction_features)} pathway interaction features:")
for feat in new_interaction_features:
    print(f"   • {feat}")

print(f"\nTotal features now: {df_features.shape[1]}")

PART 2: PATHWAY × CLINICAL INTERACTION FEATURES

Key pathways for interaction features:
  • estrogen        → Estrogen Response Early
  • proliferation   → E2F Targets
  • immune          → Interferon Gamma Response
  • glycolysis      → Glycolysis
  • hypoxia         → Hypoxia

1. ER STATUS × ESTROGEN PATHWAY INTERACTION
Distribution:
er_estrogen_interaction
ER+_LowEstrogen           1197
ER+_HighEstrogen          1116
ER-_NoEstrogen             689
Unknown                     51
ER-_UnexpectedEstrogen      22
Name: count, dtype: int64

2. GRADE × PROLIFERATION PATHWAY INTERACTION
Distribution:
grade_proliferation_interaction
Unknown                 1183
IntermediateGrade        771
HighGrade_HighProlif     479
HighGrade_LowProlif      473
LowGrade_LowProlif       121
LowGrade_HighProlif       48
Name: count, dtype: int64

3. TRIPLE NEGATIVE × GLYCOLYSIS INTERACTION
Distribution:
tn_glycolysis_interaction
NotTN                2640
TN_HighGlycolysis     231
TN_LowGlycolysis      204
Na

TypeError: '>' not supported between instances of 'str' and 'float'

In [9]:
# Feature 4: High pathway activity count (FIXED)
print("\n" + "="*70)
print("4. PATHWAY ACTIVITY SUMMARY FEATURES")

# Get ONLY original numeric pathway columns (exclude all new features)
all_new_features = ['grade_numeric', 'molecular_subtype', 'risk_group', 'node_positive', 
                    'age_group', 'treatment_combo', 'er_estrogen_interaction', 
                    'grade_proliferation_interaction', 'tn_glycolysis_interaction']

pathway_cols = [col for col in df_features.columns 
                if col not in clinical_base + imputed_vars + indicator_vars + all_new_features]

print(f"Using {len(pathway_cols)} pathway columns for activity summary")

# Calculate pathway activity counts
df_features['high_pathway_count'] = (df_features[pathway_cols] > 1.0).sum(axis=1)
df_features['low_pathway_count'] = (df_features[pathway_cols] < -1.0).sum(axis=1)
df_features['pathway_activity_ratio'] = df_features['high_pathway_count'] / (df_features['low_pathway_count'] + 1)

print(f"\nHigh pathway activity count:")
print(f"  Mean: {df_features['high_pathway_count'].mean():.2f}")
print(f"  Range: {df_features['high_pathway_count'].min():.0f} - {df_features['high_pathway_count'].max():.0f}")

print(f"\nLow pathway activity count:")
print(f"  Mean: {df_features['low_pathway_count'].mean():.2f}")
print(f"  Range: {df_features['low_pathway_count'].min():.0f} - {df_features['low_pathway_count'].max():.0f}")

print(f"\nPathway activity ratio:")
print(f"  Mean: {df_features['pathway_activity_ratio'].mean():.2f}")
print(f"  Range: {df_features['pathway_activity_ratio'].min():.2f} - {df_features['pathway_activity_ratio'].max():.2f}")

# Summary
print("\n" + "="*70)
print("PATHWAY INTERACTION FEATURES CREATED")
print("="*70)

new_interaction_features = ['er_estrogen_interaction', 'grade_proliferation_interaction', 
                            'tn_glycolysis_interaction', 'high_pathway_count', 
                            'low_pathway_count', 'pathway_activity_ratio']

print(f"\n✅ Created {len(new_interaction_features)} pathway interaction features:")
for feat in new_interaction_features:
    print(f"   • {feat}")

print(f"\nTotal features now: {df_features.shape[1]}")


4. PATHWAY ACTIVITY SUMMARY FEATURES
Using 76 pathway columns for activity summary

High pathway activity count:
  Mean: 11.23
  Range: 0 - 54

Low pathway activity count:
  Mean: 11.17
  Range: 0 - 76

Pathway activity ratio:
  Mean: 3.46
  Range: 0.00 - 54.00

PATHWAY INTERACTION FEATURES CREATED

✅ Created 6 pathway interaction features:
   • er_estrogen_interaction
   • grade_proliferation_interaction
   • tn_glycolysis_interaction
   • high_pathway_count
   • low_pathway_count
   • pathway_activity_ratio

Total features now: 113


### Part 3: Correlation Analysis & Redundancy Check

**Objective:** Identify and document highly correlated features

**Why this matters:**
- Highly correlated features (r > 0.9) add noise, not signal
- Removing redundant features improves model performance
- Reduces overfitting risk

**What we'll analyze:**
1. Pathway-pathway correlations
2. Clinical-pathway correlations
3. High correlation pairs (r > 0.9)
4. Feature variance (low variance = uninformative)

In [10]:
# Part 3: Correlation Analysis
print("="*70)
print("PART 3: CORRELATION ANALYSIS & REDUNDANCY CHECK")
print("="*70)

# Get numeric features only (exclude categorical and IDs)
numeric_features = df_features.select_dtypes(include=[np.number]).columns.tolist()

# Exclude IDs and categorical encoded features
exclude_cols = ['patient_id']
numeric_features = [col for col in numeric_features if col not in exclude_cols]

print(f"\nAnalyzing {len(numeric_features)} numeric features for correlations")

# Calculate correlation matrix
print("\nCalculating correlation matrix...")
corr_matrix = df_features[numeric_features].corr()

print(f"Correlation matrix shape: {corr_matrix.shape}")

# Find high correlations (> 0.9, excluding diagonal)
print("\n" + "="*70)
print("HIGH CORRELATION PAIRS (r > 0.9)")
print("="*70)

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False, key=abs)
    print(f"\nFound {len(high_corr_pairs)} highly correlated pairs:")
    print(high_corr_df.to_string(index=False))
else:
    print("\n✅ No highly correlated pairs found (all r < 0.9)")

# Check pathway correlations specifically
print("\n" + "="*70)
print("PATHWAY CORRELATION ANALYSIS")
print("="*70)

pathway_cols_numeric = [col for col in pathway_cols if col in numeric_features]
pathway_corr = df_features[pathway_cols_numeric].corr()

# Find highly correlated pathways
pathway_high_corr = []
for i in range(len(pathway_corr.columns)):
    for j in range(i+1, len(pathway_corr.columns)):
        if abs(pathway_corr.iloc[i, j]) > 0.8:  # Slightly lower threshold for pathways
            pathway_high_corr.append({
                'Pathway_1': pathway_corr.columns[i],
                'Pathway_2': pathway_corr.columns[j],
                'Correlation': pathway_corr.iloc[i, j]
            })

print(f"\nPathways with r > 0.8: {len(pathway_high_corr)}")
if len(pathway_high_corr) > 0:
    pathway_high_corr_df = pd.DataFrame(pathway_high_corr).sort_values('Correlation', ascending=False, key=abs)
    print("\nTop 10 highest correlated pathway pairs:")
    print(pathway_high_corr_df.head(10).to_string(index=False))

# Feature variance analysis
print("\n" + "="*70)
print("FEATURE VARIANCE ANALYSIS")
print("="*70)

feature_variance = df_features[numeric_features].var().sort_values()

print("\nLowest variance features (potential candidates for removal):")
print(feature_variance.head(10))

print("\nHighest variance features (most informative):")
print(feature_variance.tail(10))

# Summary statistics
print("\n" + "="*70)
print("CORRELATION SUMMARY")
print("="*70)

print(f"\nTotal numeric features analyzed: {len(numeric_features)}")
print(f"Highly correlated pairs (r > 0.9): {len(high_corr_pairs)}")
print(f"Pathway pairs with r > 0.8: {len(pathway_high_corr)}")

print("\n✅ Correlation analysis complete!")
print("   Most features are independent (low redundancy)")

PART 3: CORRELATION ANALYSIS & REDUNDANCY CHECK

Analyzing 93 numeric features for correlations

Calculating correlation matrix...
Correlation matrix shape: (93, 93)

HIGH CORRELATION PAIRS (r > 0.9)

Found 22 highly correlated pairs:
                Feature_1                                              Feature_2  Correlation
     lymph_nodes_positive                                    lymph_nodes_imputed     1.000000
         stage_harmonized                                          stage_imputed     1.000000
             race_missing                                     tumor_size_missing    -0.981129
              E2F Targets                                        G2-M Checkpoint     0.975894
               Cell cycle                                            E2F Targets     0.964545
               Cell cycle                                        G2-M Checkpoint     0.955303
 IL-6/JAK/STAT3 Signaling                                  Inflammatory Response     0.939647
             

### ✓ Session 2.7 Complete - Final Summary & Save

**Features created:**
- 5 clinical combinations
- 6 pathway × clinical interactions
- Total: 113 features (from 101)

**Key findings:**
- 22 highly correlated pairs identified
- Most are expected (imputed duplicates, biological co-regulation)
- Recommend removing: lymph_nodes_positive (keep imputed), stage_harmonized (keep imputed)

**Next steps:**
- Save enhanced dataset
- Document feature catalog
- Ready for final dataset preparation (Session 2.9)

In [11]:
# Part 4: Save Enhanced Dataset & Create Feature Catalog
print("="*70)
print("PART 4: SAVE ENHANCED DATASET & DOCUMENTATION")
print("="*70)

# Remove redundant features (keep imputed versions)
print("\n1. REMOVING REDUNDANT FEATURES")

features_to_remove = [
    'lymph_nodes_positive',  # Keep lymph_nodes_imputed (100% complete)
    'stage_harmonized',      # Keep stage_imputed (same thing)
    'grade_numeric'          # Temporary variable
]

print(f"\nRemoving {len(features_to_remove)} redundant features:")
for feat in features_to_remove:
    print(f"   • {feat}")

df_final = df_features.drop(columns=features_to_remove)

print(f"\nDataset shape after cleanup: {df_final.shape}")

# Create feature catalog
print("\n" + "="*70)
print("2. CREATING FEATURE CATALOG")
print("="*70)

feature_catalog = {
    'Feature_Name': [],
    'Feature_Type': [],
    'Description': [],
    'Source': []
}

# Categorize all features
for col in df_final.columns:
    feature_catalog['Feature_Name'].append(col)
    
    # Determine type and source
    if col in ['patient_id', 'cohort']:
        feature_catalog['Feature_Type'].append('Identifier')
        feature_catalog['Description'].append('Patient/cohort identifier')
        feature_catalog['Source'].append('Original')
    elif col in clinical_base:
        feature_catalog['Feature_Type'].append('Clinical_Base')
        feature_catalog['Description'].append('Original clinical variable')
        feature_catalog['Source'].append('Original')
    elif col in imputed_vars:
        feature_catalog['Feature_Type'].append('Clinical_Imputed')
        feature_catalog['Description'].append('Imputed clinical variable')
        feature_catalog['Source'].append('Session 2.6')
    elif col in indicator_vars:
        feature_catalog['Feature_Type'].append('Missing_Indicator')
        feature_catalog['Description'].append('Block-wise missing indicator')
        feature_catalog['Source'].append('Session 2.6')
    elif col in new_clinical_features:
        feature_catalog['Feature_Type'].append('Clinical_Derived')
        feature_catalog['Description'].append('Derived clinical combination')
        feature_catalog['Source'].append('Session 2.7')
    elif col in new_interaction_features:
        feature_catalog['Feature_Type'].append('Pathway_Interaction')
        feature_catalog['Description'].append('Pathway × clinical interaction')
        feature_catalog['Source'].append('Session 2.7')
    else:
        feature_catalog['Feature_Type'].append('Pathway_Score')
        feature_catalog['Description'].append('Hallmark pathway ssGSEA score')
        feature_catalog['Source'].append('Session 2.2')

catalog_df = pd.DataFrame(feature_catalog)

print(f"\nFeature breakdown by type:")
print(catalog_df['Feature_Type'].value_counts())

# Save files
print("\n" + "="*70)
print("3. SAVING FILES")
print("="*70)

# Save enhanced dataset
enhanced_path = data_dir / 'merged_dataset_enhanced.csv'
df_final.to_csv(enhanced_path, index=False)
print(f"\n✅ Saved enhanced dataset: {enhanced_path}")
print(f"   Size: {df_final.shape}")

# Save feature catalog
catalog_path = results_dir / 'tables' / 'feature_catalog.csv'
catalog_df.to_csv(catalog_path, index=False)
print(f"\n✅ Saved feature catalog: {catalog_path}")

# Save correlation summary
corr_summary_path = results_dir / 'tables' / 'high_correlations.csv'
if high_corr_pairs:
    high_corr_df.to_csv(corr_summary_path, index=False)
    print(f"\n✅ Saved correlation summary: {corr_summary_path}")

# Final summary
print("\n" + "="*70)
print("🎉 SESSION 2.7 COMPLETE!")
print("="*70)

print(f"\n📊 FINAL FEATURE COUNT: {df_final.shape[1]}")
print(f"\n📋 FEATURE BREAKDOWN:")
print(catalog_df['Feature_Type'].value_counts())

print(f"\n✅ FILES CREATED:")
print(f"   • merged_dataset_enhanced.csv ({df_final.shape[0]} × {df_final.shape[1]})")
print(f"   • feature_catalog.csv ({len(catalog_df)} features)")
print(f"   • high_correlations.csv ({len(high_corr_pairs)} pairs)")

print(f"\n⏭️  NEXT: Session 2.8 - Final Validation & Visualization")
print(f"   Estimated time: 4-6 hours")

PART 4: SAVE ENHANCED DATASET & DOCUMENTATION

1. REMOVING REDUNDANT FEATURES

Removing 3 redundant features:
   • lymph_nodes_positive
   • stage_harmonized
   • grade_numeric

Dataset shape after cleanup: (3075, 110)

2. CREATING FEATURE CATALOG

Feature breakdown by type:
Feature_Type
Pathway_Score          76
Clinical_Base          16
Pathway_Interaction     6
Clinical_Derived        5
Missing_Indicator       3
Identifier              2
Clinical_Imputed        2
Name: count, dtype: int64

3. SAVING FILES

✅ Saved enhanced dataset: D:\Projects\tcga-metabric-treatment-ai\data\merged\merged_dataset_enhanced.csv
   Size: (3075, 110)

✅ Saved feature catalog: D:\Projects\tcga-metabric-treatment-ai\results\tables\feature_catalog.csv

✅ Saved correlation summary: D:\Projects\tcga-metabric-treatment-ai\results\tables\high_correlations.csv

🎉 SESSION 2.7 COMPLETE!

📊 FINAL FEATURE COUNT: 110

📋 FEATURE BREAKDOWN:
Feature_Type
Pathway_Score          76
Clinical_Base          16
Pathway_Inter